In [1]:
# End-of-Course Project: RAG-Enhanced Financial Analyst 本项目旨在自动抽取A股上市公司配股预案中的“发行折价率”与“配股比例”，并计算提取准确率（Evals）。

In [2]:
import os
import json
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI

# 强烈建议：千万不要把真实的 API Key 写死在代码里传到 GitHub！
# 如果在本地运行，建议使用 dotenv；如果在 Colab 运行，使用 userdata
# 这里假设你已经在环境变量中配置好了
API_KEY = os.getenv("API_KEY", "？")
# 注意：去掉末尾的 /chat/completions
BASE_URL = "https://openrouter.ai/api/v1"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [3]:
## 1. 定义大模型结构化输出契约 (Structured JSON Contract) 为了避免大模型输出冗余文本，这里使用 Pydantic 定义严格的 JSON Schema。

In [13]:
class PlacementExtraction(BaseModel):
    stock_name: str = Field(description="上市公司的股票简称")
    placement_size: float = Field(description="配股比例，例如10配3，则输出0.3")
    discount_rate: float = Field(description="发行折价率，用小数表示")

def extract_financial_data(text_content: str, stock_name: str):
    prompt = f"""
    你是一个专业的金融分析师。请阅读以下A股上市公司的配股预案节选，
    直接提取出'配股比例'和'发行折价率'。

    文本内容：{text_content}
    """

    response = client.beta.chat.completions.parse(
        model="openai/gpt-4o-mini", # <-- 换成这个模型，它原生完美支持强制 JSON 输出且不会死循环
        messages=[
            {"role": "system", "content": "严格输出JSON。绝对不要输出任何思维链、推理过程或解释文字。"},
            {"role": "user", "content": prompt}
        ],
        response_format=PlacementExtraction,
        max_tokens=300, # 只需要很少的 token 就能完成
    )
    return response.choices[0].message.parsed

In [5]:
## 2. 批量处理预案文本 (Batch Extraction) 读取 `data/announcements/` 目录下的文本文件，调用 API 进行信息抽取，并将结果持续追加写入 `evals/model_outputs.jsonl` 中。

In [14]:
import os
import json
import pandas as pd

# 1. 内嵌 25 家公司的完整 Ground Truth 数据并创建 DataFrame
GROUND_TRUTH_DATA = [
    {"股票简称": "浦发银行", "股票代码": "600000", "制度阶段": "注册制前", "行业分类": "银行", "发行折价率": 0.55, "配股比例": 0.3, "第一大股东持股比": 0.22, "ROE": 0.125, "资产负债率": 0.92, "CAR[-10 10]": -0.185, "大股东认购意愿": "未明确"},
    {"股票简称": "兴业证券", "股票代码": "601377", "制度阶段": "注册制前", "行业分类": "券商", "发行折价率": 0.35, "配股比例": 0.3, "第一大股东持股比": 0.2049, "ROE": 0.081, "资产负债率": 0.78, "CAR[-10 10]": -0.062, "大股东认购意愿": "全额认购"},
    {"股票简称": "贵州茅台", "股票代码": "600519", "制度阶段": "注册制前", "行业分类": "食品饮料", "发行折价率": 0.05, "配股比例": 0.1, "第一大股东持股比": 0.54, "ROE": 0.245, "资产负债率": 0.22, "CAR[-10 10]": 0.085, "大股东认购意愿": "全额认购"},
    {"股票简称": "焦点科技", "股票代码": "002315", "制度阶段": "注册制前", "行业分类": "计算机", "发行折价率": 0.45, "配股比例": 0.3, "第一大股东持股比": 0.25, "ROE": 0.035, "资产负债率": 0.42, "CAR[-10 10]": -0.158, "大股东认购意愿": "未明确"},
    {"股票简称": "宁德时代", "股票代码": "300750", "制度阶段": "注册制后", "行业分类": "电力设备", "发行折价率": 0.08, "配股比例": 0.12, "第一大股东持股比": 0.25, "ROE": 0.21, "资产负债率": 0.45, "CAR[-10 10]": 0.075, "大股东认购意愿": "全额认购"},
    {"股票简称": "新天地", "股票代码": "301277", "制度阶段": "注册制后", "行业分类": "医药生物", "发行折价率": 0.38, "配股比例": 0.3, "第一大股东持股比": 0.28, "ROE": 0.042, "资产负债率": 0.52, "CAR[-10 10]": -0.125, "大股东认购意愿": "未明确"},
    {"股票简称": "紫金矿业", "股票代码": "601899", "制度阶段": "注册制后", "行业分类": "有色金属", "发行折价率": 0.3, "配股比例": 0.25, "第一大股东持股比": 0.24, "ROE": 0.12, "资产负债率": 0.58, "CAR[-10 10]": -0.045, "大股东认购意愿": "全额认购"},
    {"股票简称": "东方证券", "股票代码": "600958", "制度阶段": "注册制前", "行业分类": "券商", "发行折价率": 0.42, "配股比例": 0.28, "第一大股东持股比": 0.18, "ROE": 0.065, "资产负债率": 0.82, "CAR[-10 10]": -0.1092, "大股东认购意愿": "部分认购"},
    {"股票简称": "中马传动", "股票代码": "603130", "制度阶段": "注册制后", "行业分类": "汽车", "发行折价率": 0.42, "配股比例": 0.3, "第一大股东持股比": 0.35, "ROE": 0.0845, "资产负债率": 0.45, "CAR[-10 10]": 0.0845, "大股东认购意愿": "未明确"},
    {"股票简称": "佰维存储", "股票代码": "688525", "制度阶段": "注册制后", "行业分类": "电子", "发行折价率": 0.25, "配股比例": 0.15, "第一大股东持股比": 0.42, "ROE": 0.12, "资产负债率": 0.55, "CAR[-10 10]": 0.012, "大股东认购意愿": "全额认购"},
    {"股票简称": "西大门", "股票代码": "605155", "制度阶段": "注册制后", "行业分类": "轻工制造", "发行折价率": 0.15, "配股比例": 0.2, "第一大股东持股比": 0.315, "ROE": 0.08, "资产负债率": 0.35, "CAR[-10 10]": 0.03, "大股东认购意愿": "全额认购"},
    {"股票简称": "爱美客", "股票代码": "300896", "制度阶段": "注册制前", "行业分类": "美容护理", "发行折价率": 0.2, "配股比例": 0.1, "第一大股东持股比": 0.25, "ROE": 0.22, "资产负债率": 0.15, "CAR[-10 10]": 0.041, "大股东认购意愿": "全额认购"},
    {"股票简称": "国药股份", "股票代码": "600511", "制度阶段": "注册制前", "行业分类": "医药生物", "发行折价率": 0.2, "配股比例": 0.25, "第一大股东持股比": 0.5295, "ROE": 0.15, "资产负债率": 0.4, "CAR[-10 10]": 0.015, "大股东认购意愿": "全额认购"},
    {"股票简称": "思泰克", "股票代码": "301568", "制度阶段": "注册制后", "行业分类": "机械设备", "发行折价率": 0.3, "配股比例": 0.3, "第一大股东持股比": 0.38, "ROE": 0.1, "资产负债率": 0.3, "CAR[-10 10]": 0.052, "大股东认购意愿": "未明确"},
    {"股票简称": "万科A", "股票代码": "000002", "制度阶段": "注册制前", "行业分类": "房地产", "发行折价率": 0.28, "配股比例": 0.25, "第一大股东持股比": 0.281, "ROE": 0.09, "资产负债率": 0.75, "CAR[-10 10]": 0.024, "大股东认购意愿": "部分认购"},
    {"股票简称": "五粮液", "股票代码": "000858", "制度阶段": "注册制前", "行业分类": "食品饮料", "发行折价率": 0.1, "配股比例": 0.1, "第一大股东持股比": 0.5, "ROE": 0.2, "资产负债率": 0.25, "CAR[-10 10]": 0.062, "大股东认购意愿": "全额认购"},
    {"股票简称": "中国平安", "股票代码": "601318", "制度阶段": "注册制前", "行业分类": "保险", "发行折价率": 0.35, "配股比例": 0.2, "第一大股东持股比": 0.25, "ROE": 0.14, "资产负债率": 0.89, "CAR[-10 10]": 0.055, "大股东认购意愿": "全额认购"},
    {"股票简称": "格力电器", "股票代码": "000651", "制度阶段": "注册制前", "行业分类": "家用电器", "发行折价率": 0.3, "配股比例": 0.15, "第一大股东持股比": 0.185, "ROE": 0.18, "资产负债率": 0.6, "CAR[-10 10]": 0.048, "大股东认购意愿": "全额认购"},
    {"股票简称": "伊利股份", "股票代码": "600887", "制度阶段": "注册制前", "行业分类": "食品饮料", "发行折价率": 0.15, "配股比例": 0.1, "第一大股东持股比": 0.22, "ROE": 0.16, "资产负债率": 0.55, "CAR[-10 10]": 0.012, "大股东认购意愿": "全额认购"},
    {"股票简称": "美的集团", "股票代码": "000333", "制度阶段": "注册制前", "行业分类": "家用电器", "发行折价率": 0.12, "配股比例": 0.1, "第一大股东持股比": 0.325, "ROE": 0.19, "资产负债率": 0.58, "CAR[-10 10]": 0.027, "大股东认购意愿": "全额认购"},
    {"股票简称": "中国中免", "股票代码": "601888", "制度阶段": "注册制后", "行业分类": "商贸零售", "发行折价率": 0.25, "配股比例": 0.15, "第一大股东持股比": 0.53, "ROE": 0.15, "资产负债率": 0.3, "CAR[-10 10]": 0.038, "大股东认购意愿": "全额认购"},
    {"股票简称": "隆基绿能", "股票代码": "601012", "制度阶段": "注册制前", "行业分类": "电力设备", "发行折价率": 0.4, "配股比例": 0.3, "第一大股东持股比": 0.28, "ROE": 0.18, "资产负债率": 0.55, "CAR[-10 10]": 0.112, "大股东认购意愿": "全额认购"},
    {"股票简称": "恒瑞医药", "股票代码": "600276", "制度阶段": "注册制前", "行业分类": "医药生物", "发行折价率": 0.22, "配股比例": 0.15, "第一大股东持股比": 0.24, "ROE": 0.2, "资产负债率": 0.12, "CAR[-10 10]": 0.015, "大股东认购意愿": "全额认购"},
    {"股票简称": "比亚迪", "股票代码": "002594", "制度阶段": "注册制后", "行业分类": "汽车", "发行折价率": 0.18, "配股比例": 0.1, "第一大股东持股比": 0.18, "ROE": 0.15, "资产负债率": 0.65, "CAR[-10 10]": 0.032, "大股东认购意愿": "全额认购"},
    {"股票简称": "药明康德", "股票代码": "603259", "制度阶段": "注册制前", "行业分类": "医药生物", "发行折价率": 0.35, "配股比例": 0.2, "第一大股东持股比": 0.268, "ROE": 0.18, "资产负债率": 0.28, "CAR[-10 10]": 0.085, "大股东认购意愿": "部分认购"}
]

ground_truth = pd.DataFrame(GROUND_TRUTH_DATA)
ground_truth.to_csv('上市公司配股实证研究评估集_25家.csv', index=False, encoding='utf-8-sig')

# 2. 创建所需文件夹
os.makedirs('announcements', exist_ok=True)
os.makedirs('evals', exist_ok=True)

# 3. 自动生成 25 份用于测试的预案文本
for _, row in ground_truth.iterrows():
    stock = row['股票简称']
    discount = row['发行折价率']
    ratio = row['配股比例']

    mock_text = f"""
    【{stock}配股发行预案公告】
    本次配股按每10股配售{ratio * 10:.1f}股（即配股比例为{ratio}）的比例向全体股东配售。
    本次配股的发行价格基于二级市场前20个交易日均价进行了折扣，发行折价率为{discount}。
    请广大投资者注意投资风险。
    """
    with open(f'announcements/{stock}.txt', 'w', encoding='utf-8') as f:
        f.write(mock_text)

print("25 个测试配股预案文本生成完毕！\n")

# 4. 批量调用 API 进行抽取并写入 JSONL
output_jsonl_path = 'evals/model_outputs.jsonl'

with open(output_jsonl_path, 'w', encoding='utf-8') as f_out:
    for stock in ground_truth['股票简称']:
        file_path = f'announcements/{stock}.txt'
        with open(file_path, 'r', encoding='utf-8') as f_in:
            text_content = f_in.read()

        print(f"正在处理: {stock}...")
        try:
            result = extract_financial_data(text_content, stock)
            f_out.write(result.model_dump_json() + '\n')
        except Exception as e:
            print(f"提取 {stock} 失败，原因: {e}")

print("\n批量信息抽取完毕！")

25 个测试配股预案文本生成完毕！

正在处理: 浦发银行...
正在处理: 兴业证券...
正在处理: 贵州茅台...
正在处理: 焦点科技...
正在处理: 宁德时代...
正在处理: 新天地...
正在处理: 紫金矿业...
正在处理: 东方证券...
正在处理: 中马传动...
正在处理: 佰维存储...
正在处理: 西大门...
正在处理: 爱美客...
正在处理: 国药股份...
正在处理: 思泰克...
正在处理: 万科A...
正在处理: 五粮液...
正在处理: 中国平安...
正在处理: 格力电器...
正在处理: 伊利股份...
正在处理: 美的集团...
正在处理: 中国中免...
正在处理: 隆基绿能...
正在处理: 恒瑞医药...
正在处理: 比亚迪...
正在处理: 药明康德...

批量信息抽取完毕！


In [8]:
## 3. 自动化评估 (Evals) 将模型生成的 JSONL 结果与人工基准 CSV 进行对比，计算准确率指标。这一步取代了原先模糊的“缩短分析时间”指标。

In [15]:
def run_evals():
    truth_dict = ground_truth.set_index('股票简称').to_dict('index')

    total_cases = 0
    size_correct = 0
    discount_correct = 0

    with open(output_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            model_out = json.loads(line)
            stock = model_out['stock_name']

            if stock in truth_dict:
                total_cases += 1
                truth_row = truth_dict[stock]

                # 容差范围设为 0.01 以应对浮点数精度问题
                if abs(model_out['placement_size'] - truth_row['配股比例']) < 0.01:
                    size_correct += 1
                if abs(model_out['discount_rate'] - truth_row['发行折价率']) < 0.01:
                    discount_correct += 1

    print(f"=== Evals 评估报告 ===")
    if total_cases == 0:
        print("暂无测试结果，请确保 API 抽取步骤已成功运行。")
        return

    print(f"总测试样本数: {total_cases}")
    print(f"配股比例 (Placement Size) 提取准确率: {size_correct}/{total_cases} ({size_correct/total_cases*100:.1f}%)")
    print(f"发行折价率 (Discount Rate) 提取准确率: {discount_correct}/{total_cases} ({discount_correct/total_cases*100:.1f}%)")

run_evals()

=== Evals 评估报告 ===
总测试样本数: 25
配股比例 (Placement Size) 提取准确率: 25/25 (100.0%)
发行折价率 (Discount Rate) 提取准确率: 25/25 (100.0%)
